In [6]:
## automatically reload any modules read below that might have changed (e.g. plots)
%reload_ext autoreload
%autoreload 2

In [7]:
# importing plotting and locus tools: 
import neoom as nf
import NEOMOD3 as nm3  

In [8]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from urllib.request import urlopen
from io import StringIO
import numpy as np
import astropy.units as u
from astropy.time import Time
from numpy import sin, cos
from astropy.coordinates import EarthLocation, get_body_barycentric_posvel, solar_system_ephemeris 
from astropy.coordinates import SkyCoord, get_sun, GCRS, GeocentricTrueEcliptic
import pyarrow as pa
import adam_core.coordinates
from adam_core.time import Timestamp
from adam_core.coordinates import CartesianCoordinates, Origin
from adam_core.constants import KM_P_AU, S_P_DAY
from adam_core.orbits import Orbits
from adam_core.utils import get_perturber_state
from adam_core.coordinates.origin import OriginCodes
from scipy.interpolate import RegularGridInterpolator


In [9]:
df, array4D, H_center, a_center, e_center, i_center = nm3.getNEOMOD3orbits()

In [10]:
obstime_str="2025-09-23T00:00:00"
#ra_deg, dec_deg, dra_deg_per_day, ddec_deg_per_day = 186.35, -5.0, 46.27, -19.0
ra_deg, dec_deg, dra_deg_per_day, ddec_deg_per_day = 10.0, 20.0, 0.12, 0.22 
# add mag here
mag = 20.0
# anti-Sun at Sep 23 = equinox so RA = 0, Dec = 0
ra0_deg, dec0_deg = 0.0, 0.0

In [11]:

l_hat, v_hat = nf.compute_unit_vectors(ra_deg, dec_deg, dra_deg_per_day, ddec_deg_per_day)
obstime, rE, vE, r_obs = nf.get_earth_and_observer(obstime_str) 
neo_score, a, e, i = nf.compute_neomod3_weight(0.2, 3.0, l_hat, v_hat, obstime, rE, vE, r_obs, array4D, H_center, a_center, e_center, i_center)




print("neo_score:", neo_score)
print("a:", a)
print("e:", e)
print("i:", i)

neo_score: 2.2396
a: 1.8147948989466378
e: 0.358504361404317
i: 26.36952675502365


In [12]:
from astropy.coordinates import SkyCoord, GeocentricTrueEcliptic
import astropy.units as u
from astropy.time import Time

def ecl_to_radec(lambda_deg, beta_deg, obstime_str):
    obstime = Time(obstime_str)
    ecl = SkyCoord(lon=lambda_deg*u.deg, lat=beta_deg*u.deg,
                   frame=GeocentricTrueEcliptic(equinox=obstime))
    icrs = ecl.icrs
    return icrs.ra.deg, icrs.dec.deg

In [13]:
def comp_sum2(ra_deg, dec_deg, *, N_rate=51, vlam_min=-0.6, vlam_max=0.6, vbet_min=-0.6, vbet_max=0.6):
    
    vlam_vals = np.linspace(vlam_min, vlam_max, N_rate)
    vbet_vals = np.linspace(vbet_min, vbet_max, N_rate)
    K = np.zeros((len(vlam_vals), len(vbet_vals)), dtype=float)

    for i, vlam in enumerate(vlam_vals):
        for j, vbet in enumerate(vbet_vals):
            
            dra, ddec = nf.ecliptic_rates_to_radec_rates(ra_deg, dec_deg, vlam, vbet)
            w_int = nf.weight_marginalized_over_d_and_ddot(array4D=array4D, H_center=H_center, a_center=a_center, e_center=e_center, i_center=i_center,
                                                        ra_deg=ra0_deg, dec_deg=dec0_deg, dra_deg_per_day=dra, ddec_deg_per_day=ddec,obstime_str=obstime_str
                                                        )
            
            K[j, i] = w_int

    extent = [vlam_vals[0], vlam_vals[-1], vbet_vals[0], vbet_vals[-1]]
    return K, extent


In [46]:
# beta is fixed, lambda goes from 0 to 360
beta_deg = 0.0
lambda_list = np.arange(0, 360, 10)  # every 15 degrees
targets = [(lam, beta_deg) for lam in lambda_list]

#beta_list = np.arange(-20, 21, 15)
#targets = [(lam, beta) for beta in beta_list for lam in lambda_list]

all_frames = [] #store each plot for the gif
Kmins, Kmaxs = [], [] #stores min/max

for lam, beta in targets:
    ra_deg, dec_deg = ecl_to_radec(lam, beta, obstime_str)
    K, extent = comp_sum2(ra_deg, dec_deg, N_rate=141,
                          vlam_min=-0.6, vlam_max=0.6,
                          vbet_min=-0.6, vbet_max=0.6)
    all_frames.append({"lam": lam, "beta": beta, "ra": ra_deg, "dec": dec_deg,
                       "K": K, "extent": extent})
    Kmins.append(np.nanmin(K))
    Kmaxs.append(np.nanmax(K))


#vmin = np.percentile(np.ravel([f["K"] for f in all_frames]), 1)
vmin = 0.0
vmax = np.percentile(np.ravel([f["K"] for f in all_frames]), 99)


In [31]:
# Choose a frame index to inspect, e.g. the 0th frame
frame = all_frames[0]

K = frame["K"]
extent = frame["extent"]

# Rebuild coordinate arrays matching K shape
vlam_vals = np.linspace(extent[0], extent[1], K.shape[1])   # x-axis
vbet_vals = np.linspace(extent[2], extent[3], K.shape[0])   # y-axis



In [33]:
vmin = np.percentile(np.ravel([f["K"] for f in all_frames]), 1)
vmax = np.percentile(np.ravel([f["K"] for f in all_frames]), 99)


In [35]:
white_mask = (K < vmin) | ~np.isfinite(K)


In [36]:
js, is_ = np.where(white_mask)

print("White pixel count:", len(js))

# Print the first few
for j, i in list(zip(js, is_))[:10]:
    print(f"vλ={vlam_vals[i]:+.3f},  vβ={vbet_vals[j]:+.3f},  K={K[j,i]:.4g}")


White pixel count: 75
vλ=-0.133,  vβ=+0.200,  K=1.992
vλ=-0.133,  vβ=+0.213,  K=1.986
vλ=-0.133,  vβ=+0.227,  K=1.994
vλ=-0.133,  vβ=+0.240,  K=1.994
vλ=-0.133,  vβ=+0.253,  K=1.989
vλ=-0.133,  vβ=+0.267,  K=1.989
vλ=-0.133,  vβ=+0.280,  K=1.985
vλ=-0.133,  vβ=+0.293,  K=1.985
vλ=-0.133,  vβ=+0.307,  K=1.983
vλ=-0.133,  vβ=+0.320,  K=1.982


In [47]:
import os
from matplotlib import colormaps as cm  

out_dir = "kmap_frames_ecliptic_sweep9"
os.makedirs(out_dir, exist_ok=True)
cmap = cm.get_cmap("viridis").with_extremes(under="white")

def save_frame(frame, idx, vmin, vmax):
    K, extent = frame["K"], frame["extent"]
    lam, beta = frame["lam"], frame["beta"]
    ra, dec  = frame["ra"], frame["dec"]

    fig, ax = plt.subplots(figsize=(6.0, 6.0), dpi=160)
    im = ax.imshow(K, origin="lower", extent=extent, cmap=cmap,
                   vmin=vmin, vmax=vmax, aspect="equal")
    plt.colorbar(im, ax=ax, label="NEOMOD3 weight (marginalized over d, ddot)")
    ax.axhline(0, color="0.8", lw=1); ax.axvline(0, color="0.8", lw=1)
    ax.set_xlabel(r"$v_\lambda$ (deg/day)")
    ax.set_ylabel(r"$v_\beta$ (deg/day)")
    ax.set_title(f"λ={lam:.0f}°, β={beta:.1f}°  →  RA={ra:.2f}°, Dec={dec:.2f}°")

    fname = os.path.join(out_dir, f"frame_{idx:03d}_lam{lam:03.0f}_beta{beta:+.1f}.png")
    plt.tight_layout()
    plt.savefig(fname, bbox_inches="tight")
    plt.close(fig)
    return fname

pngs = [save_frame(f, i, vmin, vmax) for i, f in enumerate(all_frames)]
print(f"Saved {len(pngs)} frames to {out_dir}")


Saved 36 frames to kmap_frames_ecliptic_sweep9


In [48]:
import imageio.v2 as imageio
gif_path = "kmap_ecliptic_sweep9.gif"
imageio.mimsave(gif_path, [imageio.imread(p) for p in pngs], duration=3)
print("GIF:", gif_path)


GIF: kmap_ecliptic_sweep9.gif


In [29]:
import imageio.v2 as imageio

# using PNGs -> slow by increasing duration
imageio.mimsave("kmap2.gif",
                [imageio.imread(p) for p in pngs],
                duration=2.0,      # 2 s per frame (bigger = slower)
                loop=2)            # 0=infinite loop; set 1 to play once


In [45]:
import numpy as np, time


def one_call():
    ra_deg, dec_deg = ra0_deg, dec0_deg
    dra, ddec = nf.ecliptic_rates_to_radec_rates(ra_deg, dec_deg, 0.0, 0.0)
    return nf.weight_marginalized_over_d_and_ddot(
        array4D=array4D, H_center=H_center, a_center=a_center,
        e_center=e_center, i_center=i_center,
        ra_deg=ra_deg, dec_deg=dec_deg,
        dra_deg_per_day=dra, ddec_deg_per_day=ddec,obstime_str=obstime_str
    )

# warmup
for _ in range(5): one_call()

# timing
N = 50
t0 = time.perf_counter()
for _ in range(N): one_call()
t1 = time.perf_counter()
t_call = (t1 - t0) / N  # seconds
print(f"~{t_call*1000:.2f} ms per weight eval")

# estimated total (adjust N_targets and N_rate to your sweep)
N_targets = 36
N_rate = 141
total_calls = N_targets * (N_rate**2)
eta_sec = total_calls * t_call
print(f"Estimated compute time: ~{eta_sec/60:.1f} minutes for the first pass")


~101.53 ms per weight eval
Estimated compute time: ~1211.1 minutes for the first pass
